<a href="https://colab.research.google.com/github/Heng1222/VeriPromiseESG_2026_TEAM_9906/blob/feat-model-train/app/model/model_setfit_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SetFit Inference for VeriPromiseESG

This notebook mirrors the original `model_inference.ipynb` interface and CSV output format, but loads SetFit artifacts generated by `model_setfit.ipynb`.

Expected model artifact layout:

```text
setfit_outputs/
├── setfit_thresholds.json
├── fold_1/
│   ├── encoder/
│   └── task_heads.joblib
├── ...
└── fold_5/
    ├── encoder/
    └── task_heads.joblib
```

Input test CSV columns must include `id`, `data`, and `esg_type`. Output columns are exactly:

```text
id,promise_status,verification_timeline,evidence_status,evidence_quality
```


In [1]:
# Install inference dependencies. Restart runtime/kernel if Colab asks for it.
# !pip install -q sentence-transformers scikit-learn pandas numpy tqdm joblib huggingface_hub


In [2]:
import gc
import json
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm


In [3]:
# ==========================================
# 0. Configuration
# ==========================================

RAW_BASE_URL = "https://raw.githubusercontent.com/Heng1222/VeriPromiseESG_2026_TEAM_9906/main/app/data/clean_data/"

# Match the original inference notebook default: use val_fold_1 as a smoke-test target.
# Replace this with the real test CSV path when exporting a real submission.
TEST_CSV_PATH = f"{RAW_BASE_URL}val_fold_1.csv"

# Local/Colab artifact directory produced by model_setfit.ipynb.
SETFIT_MODEL_DIR = Path("")

# Set this to the SetFit Hub repo created by model_setfit.ipynb.
# The default follows the original repo naming convention; change it if your upload used another repo id.
DEFAULT_REPO_ID = "maxbeettww/VeriPromise_ESG_2026_9906_SetFit"

FOLDS = [1, 2, 3, 4, 5]
MAX_SEQ_LENGTH = 512
HEAD_RATIO = 0.25
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DEFAULT_T1_THRESHOLD = 0.5
DEFAULT_T3_THRESHOLD = 0.5

print(f"Device: {DEVICE}")


Device: cuda


In [4]:
# ==========================================
# 1. Shared constants and text builder
# Must match model_setfit.ipynb exactly.
# ==========================================

ID_COLUMN = "id"
TEXT_COLUMN = "data"
ESG_COLUMN = "esg_type"

TARGET_COLUMNS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]

TASK_PREFIX = {
    "t1": "任務：判斷是否有 ESG 承諾。",
    "t2": "任務：判斷承諾驗證時間。",
    "t3": "任務：判斷是否提供證據。",
    "t4": "任務：判斷證據品質。",
}

TASK_CLASSES = {
    "t1": ["No", "Yes"],
    "t2": ["already", "within_2_years", "between_2_and_5_years", "longer_than_5_years"],
    "t3": ["No", "Yes"],
    "t4": ["Clear", "Not Clear", "Misleading"],
}


def process_esg_type(esg_value):
    if pd.isna(esg_value) or str(esg_value).strip() == "":
        return "未知"
    parts = [part.strip() for part in str(esg_value).split(";") if part.strip()]
    return "、".join(parts) if parts else "未知"


def truncate_text_by_tokens(text, tokenizer, max_seq_length=MAX_SEQ_LENGTH, head_ratio=HEAD_RATIO):
    """Head-tail truncation without calling tokenizer.encode on overlong text.

    This mirrors model_setfit.ipynb and avoids tokenizer warnings/errors like
    "Token indices sequence length is longer than ..." before truncation.
    """
    text = str(text)
    max_body_len = max_seq_length - 2
    tokens = tokenizer.tokenize(text)
    if len(tokens) <= max_body_len:
        return text

    head_len = int(max_body_len * head_ratio)
    tail_len = max_body_len - head_len
    kept_tokens = tokens[:head_len] + tokens[-tail_len:]
    if hasattr(tokenizer, "convert_tokens_to_string"):
        return tokenizer.convert_tokens_to_string(kept_tokens)

    kept_ids = tokenizer.convert_tokens_to_ids(kept_tokens)
    return tokenizer.decode(kept_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)


def build_base_text(row, tokenizer=None):
    esg_text = process_esg_type(row.get(ESG_COLUMN, ""))
    raw_text = str(row.get(TEXT_COLUMN, ""))
    full_text = f"ESG類型：{esg_text}。文本：{raw_text}"
    if tokenizer is not None:
        full_text = truncate_text_by_tokens(full_text, tokenizer)
    return full_text


def build_task_text(task_key, row, tokenizer=None):
    return TASK_PREFIX[task_key] + build_base_text(row, tokenizer=tokenizer)


def validate_test_csv(df):
    required = [ID_COLUMN, TEXT_COLUMN, ESG_COLUMN]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Test CSV missing required columns: {missing}")
    if df[ID_COLUMN].duplicated().any():
        duplicated_ids = df.loc[df[ID_COLUMN].duplicated(), ID_COLUMN].tolist()
        raise ValueError(f"Test CSV contains duplicated ids: {duplicated_ids[:10]}")


In [5]:
# ==========================================
# 2. Artifact loading
# ==========================================

def has_local_setfit_artifacts(model_root):
    model_root = Path(model_root)
    for fold in FOLDS:
        fold_dir = model_root / f"fold_{fold}"
        if not (fold_dir / "encoder").exists():
            return False
        if not (fold_dir / "task_heads.joblib").exists():
            return False
    return True


def find_artifact_root(root):
    root = Path(root)
    candidates = [root, root / "setfit_outputs"]
    for candidate in candidates:
        if has_local_setfit_artifacts(candidate):
            return candidate
    return None


def resolve_model_root(repo_id=None, model_dir=SETFIT_MODEL_DIR):
    model_dir = Path(model_dir)
    if repo_id:
        print(f"Downloading SetFit artifacts from Hub repo: {repo_id}")
        downloaded = snapshot_download(
            repo_id=repo_id,
            allow_patterns=[
                "setfit_thresholds.json",
                "setfit_inference_config.json",
                "README.md",
                "fold_*/encoder/**",
                "fold_*/task_heads.joblib",
                "setfit_outputs/setfit_thresholds.json",
                "setfit_outputs/setfit_inference_config.json",
                "setfit_outputs/README.md",
                "setfit_outputs/fold_*/encoder/**",
                "setfit_outputs/fold_*/task_heads.joblib",
            ],
        )
        downloaded = Path(downloaded)
        hub_root = find_artifact_root(downloaded)
        if hub_root is None:
            raise FileNotFoundError(
                f"Downloaded repo does not contain expected setfit_outputs layout: {downloaded}"
            )
        return hub_root

    local_root = find_artifact_root(model_dir)
    if local_root is not None:
        print(f"Using local SetFit artifacts: {local_root.resolve()}")
        return local_root

    raise FileNotFoundError(
        "SetFit artifacts were not found. Upload the setfit_outputs folder to Colab, "
        "or pass a Hugging Face repo_id containing fold_*/encoder and fold_*/task_heads.joblib."
    )


def apply_inference_config(model_root):
    global TASK_PREFIX, TASK_CLASSES, MAX_SEQ_LENGTH, HEAD_RATIO
    config_path = Path(model_root) / "setfit_inference_config.json"
    if not config_path.exists():
        print("No setfit_inference_config.json found; using notebook defaults.")
        return {}
    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)
    TASK_PREFIX = config.get("task_prefix", TASK_PREFIX)
    TASK_CLASSES = config.get("task_classes", TASK_CLASSES)
    MAX_SEQ_LENGTH = int(config.get("max_seq_length", MAX_SEQ_LENGTH))
    HEAD_RATIO = float(config.get("head_ratio", HEAD_RATIO))
    print(f"Loaded inference config from {config_path}")
    return config


def load_thresholds(model_root, config=None):
    threshold_path = Path(model_root) / "setfit_thresholds.json"
    if threshold_path.exists():
        with open(threshold_path, "r", encoding="utf-8") as f:
            thresholds = json.load(f)
        return {
            "t1_threshold": float(thresholds.get("t1_threshold", DEFAULT_T1_THRESHOLD)),
            "t3_threshold": float(thresholds.get("t3_threshold", DEFAULT_T3_THRESHOLD)),
        }
    if config and "thresholds" in config:
        thresholds = config["thresholds"]
        return {
            "t1_threshold": float(thresholds.get("t1_threshold", DEFAULT_T1_THRESHOLD)),
            "t3_threshold": float(thresholds.get("t3_threshold", DEFAULT_T3_THRESHOLD)),
        }
    return {"t1_threshold": DEFAULT_T1_THRESHOLD, "t3_threshold": DEFAULT_T3_THRESHOLD}


def load_fold_artifacts(model_root, fold):
    fold_dir = Path(model_root) / f"fold_{fold}"
    encoder = SentenceTransformer(str(fold_dir / "encoder"), device=DEVICE)
    encoder.max_seq_length = MAX_SEQ_LENGTH
    heads = joblib.load(fold_dir / "task_heads.joblib")
    return encoder, heads


In [6]:
# ==========================================
# 3. Probability prediction and 5-fold ensemble
# ==========================================

def encode_task_texts(encoder, df, task_key, batch_size=BATCH_SIZE):
    texts = [build_task_text(task_key, row, tokenizer=encoder.tokenizer) for _, row in df.iterrows()]
    return encoder.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )


def predict_task_probabilities(encoder, heads, df, task_key):
    embeddings = encode_task_texts(encoder, df, task_key)
    head = heads[task_key]
    classifier = head["classifier"]
    label_encoder = head["label_encoder"]
    raw_probs = classifier.predict_proba(embeddings)

    class_probs = pd.DataFrame(0.0, index=df.index, columns=TASK_CLASSES[task_key])
    for encoded_index, label in enumerate(label_encoder.classes_):
        if label in class_probs.columns:
            class_probs[label] = raw_probs[:, encoded_index]
    return class_probs


def predict_all_probabilities_for_fold(model_root, fold, test_df):
    encoder, heads = load_fold_artifacts(model_root, fold)
    output = pd.DataFrame({ID_COLUMN: test_df[ID_COLUMN].values})

    for task_key in ["t1", "t2", "t3", "t4"]:
        probs = predict_task_probabilities(encoder, heads, test_df.reset_index(drop=True), task_key)
        for class_name in TASK_CLASSES[task_key]:
            output[f"{task_key}__{class_name}"] = probs[class_name].values

    del encoder, heads
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return output


def average_probability_frames(probability_frames):
    base = probability_frames[0][[ID_COLUMN]].copy()
    prob_cols = [col for col in probability_frames[0].columns if col != ID_COLUMN]
    for col in prob_cols:
        base[col] = np.mean([frame[col].to_numpy() for frame in probability_frames], axis=0)
    return base


In [7]:
# ==========================================
# 4. Routing and CSV export
# ==========================================

def argmax_from_probs(row, task_key):
    classes = TASK_CLASSES[task_key]
    values = np.array([row[f"{task_key}__{class_name}"] for class_name in classes], dtype=float)
    return classes[int(values.argmax())]


def route_predictions(prob_df, t1_threshold=DEFAULT_T1_THRESHOLD, t3_threshold=DEFAULT_T3_THRESHOLD):
    results = []
    for _, row in prob_df.iterrows():
        t1_pred = "Yes" if float(row["t1__Yes"]) >= t1_threshold else "No"

        if t1_pred == "No":
            results.append(
                {
                    ID_COLUMN: row[ID_COLUMN],
                    "promise_status": "No",
                    "verification_timeline": "N/A",
                    "evidence_status": "N/A",
                    "evidence_quality": "N/A",
                }
            )
            continue

        t2_pred = argmax_from_probs(row, "t2")
        t3_pred = "Yes" if float(row["t3__Yes"]) >= t3_threshold else "No"

        if t3_pred == "No":
            results.append(
                {
                    ID_COLUMN: row[ID_COLUMN],
                    "promise_status": "Yes",
                    "verification_timeline": t2_pred,
                    "evidence_status": "No",
                    "evidence_quality": "N/A",
                }
            )
            continue

        t4_pred = argmax_from_probs(row, "t4")
        results.append(
            {
                ID_COLUMN: row[ID_COLUMN],
                "promise_status": "Yes",
                "verification_timeline": t2_pred,
                "evidence_status": "Yes",
                "evidence_quality": t4_pred,
            }
        )

    return pd.DataFrame(results)[[ID_COLUMN] + TARGET_COLUMNS]


def validate_output_csv(output_df, test_df):
    expected_columns = [ID_COLUMN] + TARGET_COLUMNS
    if list(output_df.columns) != expected_columns:
        raise ValueError(f"Unexpected output columns: {list(output_df.columns)}")
    if len(output_df) != len(test_df):
        raise ValueError(f"Row count mismatch: output={len(output_df)}, input={len(test_df)}")
    if output_df[ID_COLUMN].tolist() != test_df[ID_COLUMN].tolist():
        raise ValueError("Output id order differs from input CSV.")


def ensemble_inference_and_export(repo_id, test_csv_path, output_csv_path="final_submission.csv", model_dir=SETFIT_MODEL_DIR):
    print("Starting SetFit 5-fold soft-voting inference...")
    model_root = resolve_model_root(repo_id=repo_id, model_dir=model_dir)
    inference_config = apply_inference_config(model_root)
    thresholds = load_thresholds(model_root, config=inference_config)
    print(f"Using thresholds: {thresholds}")

    test_df = pd.read_csv(test_csv_path)
    validate_test_csv(test_df)
    test_df = test_df.reset_index(drop=True)

    probability_frames = []
    for fold in FOLDS:
        print(f"\nLoading Fold {fold} SetFit artifacts...")
        fold_probs = predict_all_probabilities_for_fold(model_root, fold, test_df)
        probability_frames.append(fold_probs)

    avg_probs = average_probability_frames(probability_frames)
    final_output = route_predictions(
        avg_probs,
        t1_threshold=thresholds["t1_threshold"],
        t3_threshold=thresholds["t3_threshold"],
    )

    final_output[ID_COLUMN] = test_df[ID_COLUMN].values
    final_output = final_output[[ID_COLUMN] + TARGET_COLUMNS]
    validate_output_csv(final_output, test_df)
    final_output.to_csv(output_csv_path, index=False)
    print(f"Inference complete. Exported prediction CSV to: {output_csv_path}")
    print(final_output.head())
    return final_output


In [8]:
# ==========================================
# 5. Run inference
# ==========================================
# Local/Colab artifacts generated by model_setfit.ipynb:
test_url = f"../data/ori_data/vpesg4k_val_1000.csv"
# ensemble_inference_and_export(DEFAULT_REPO_ID, test_url, "final_submission.csv", model_dir=SETFIT_MODEL_DIR)

# If your SetFit artifacts are uploaded to Hugging Face Hub, use:
ensemble_inference_and_export("maxbeettww/VeriPromise_ESG_2026_9906_SetFit", test_url, "final_submission.csv")

# If your real test file is uploaded to Colab, use for example:
# ensemble_inference_and_export(DEFAULT_REPO_ID, "/content/test.csv", "final_submission.csv", model_dir=SETFIT_MODEL_DIR)


Starting SetFit 5-fold soft-voting inference...


Fetching 53 files:   0%|          | 0/53 [00:00<?, ?it/s]

Loaded inference config from /home/public/.cache/huggingface/hub/models--maxbeettww--VeriPromise_ESG_2026_9906_SetFit/snapshots/a6f2d22e990416b478a0988c92459b074ff199bc/setfit_inference_config.json
Using thresholds: {'t1_threshold': 0.2, 't3_threshold': 0.3}

Loading Fold 1 SetFit artifacts...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (595 > 512). Running this sequence through the model will result in indexing errors


...


Loading Fold 5 SetFit artifacts...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (595 > 512). Running this sequence through the model will result in indexing errors


Inference complete. Exported prediction CSV to: final_submission.csv
      id promise_status  verification_timeline evidence_status  \
0  11001            Yes                already             Yes   
1  11002            Yes  between_2_and_5_years             Yes   
2  11003            Yes                already             Yes   
3  11004            Yes                already             Yes   
4  11005             No                    N/A             N/A   

  evidence_quality  
0            Clear  
1            Clear  
2            Clear  
3            Clear  
4              N/A  


,id,promise_status,verification_timeline,evidence_status,evidence_quality
0,11001,Yes,already,Yes,Clear
1,11002,Yes,between_2_and_5_years,Yes,Clear
2,11003,Yes,already,Yes,Clear
3,11004,Yes,already,Yes,Clear
4,11005,No,N/A,N/A,N/A
...,...,...,...,...,...
995,11996,Yes,already,Yes,Clear
996,11997,Yes,already,Yes,Clear
997,11998,Yes,longer_than_5_years,Yes,Clear
998,11999,Yes,already,Yes,Clear
